# 3-branch (3D + 2x2D, CrossGate) trained on **BM3D-denoised full 200^3** - Kaggle

Trains the frozen 3-branch model (`scripts.controls_model.ControlsModel`: ResNeXt3D + two
2D branches on `slab_mip` / `aip_full`, fused by CrossGate) on the **BM3D-denoised**
Harvard-GF volumes at full resolution 200^3.

## Kaggle setup
1. Notebook settings -> **Internet: On** (required for HF + W&B).
2. Add-ons -> Secrets: `HF_TOKEN` and `WANDB_API_KEY` (both required).
3. Accelerator: **GPU T4 x2** or **P100** (single GPU is used; DDP is not supported).
4. Disk: consolidated BM3D-200 volumes need ~27 GB under `/kaggle/temp`; the cell below
   checks free space first. If Kaggle disk is insufficient, create a private Kaggle Dataset
   from the HF files and attach it, then set `B3_DATA_DIR` to the attached folder.

## Data source
- BM3D volumes: `tqhuyen/harvard-gf-denoise-benchmark-v2`, prefix
  `classical/bm3d/3375a321513938835d2c/volumes/{split}/shard-*.npy` (1 GB shards, uint8 200^3).
- Labels: `tqhuyen/harvard-oct-glaucoma-200` (`{split}_labels.npy`).

## Memory warning (full 200^3)
Full-resolution 3D activations are large; the notebook probes batch size, uses AMP, gradient
accumulation and **optional gradient checkpointing** (`B3_GRAD_CKPT=1`, default on). If the
probe still fails, lower `B3_ENC3D` (e.g. `16,32,64,128`) or use `B3_RES3D=128`.


In [ ]:
import os, sys, subprocess, json, math, time, tempfile, hashlib
from pathlib import Path
SMOKE = os.environ.get('B3_SMOKE', '0') == '1'
IN_KAGGLE = Path('/kaggle').is_dir()
REPO = Path('/kaggle/working/glaucoma-thesis') if IN_KAGGLE else Path.cwd()
if not SMOKE and IN_KAGGLE:
    if not REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Tqhuyen/glaucoma-thesis.git', str(REPO)], check=True)
    def _have(_mod):
        try:
            __import__(_mod)
            return True
        except Exception:
            return False
    _need = [pkg for mod, pkg in (('timm', 'timm==1.0.29'), ('wandb', 'wandb'),
                                  ('huggingface_hub', 'huggingface_hub'), ('dotenv', 'python-dotenv'),
                                  ('sklearn', 'scikit-learn'), ('skimage', 'scikit-image'),
                                  ('psutil', 'psutil'), ('filelock', 'filelock')) if not _have(mod)]
    if _need:
        print('[setup] installing (no torch upgrade):', _need, flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps'] + _need, check=True)
sys.path.insert(0, str(REPO))
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from scripts import final_model as fm, final_training as ft, controls_model as cm, controls_training as ct
ft.load_env_file()
if not SMOKE:
    try:
        from kaggle_secrets import UserSecretsClient
        _us = UserSecretsClient()
        for _k in ('HF_TOKEN', 'WANDB_API_KEY'):
            _v = _us.get_secret(_k)
            if _v:
                os.environ[_k] = _v
    except Exception as _e:
        print('[secrets] kaggle_secrets unavailable:', _e)
if SMOKE:
    os.environ.setdefault('WANDB_MODE', 'offline')
    os.environ.setdefault('WANDB_API_KEY', 'local')
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')
DEVICE = torch.device('cpu' if SMOKE else ('cuda' if torch.cuda.is_available() else 'cpu'))
USE_AMP = DEVICE.type == 'cuda'
if USE_AMP:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
print('[setup] device', DEVICE, '| kaggle', IN_KAGGLE, '| smoke', SMOKE)
if DEVICE.type == 'cuda':
    _cap = torch.cuda.get_device_capability()
    _sm = f'sm_{_cap[0]}{_cap[1]}'
    _arch = torch.cuda.get_arch_list()
    print('[gpu]', torch.cuda.get_device_name(0), '| capability', _sm,
          '| torch', torch.__version__, '| cuda', torch.version.cuda)
    print('[gpu] torch compiled archs:', _arch)
    if _sm not in _arch and f'compute_{_cap[0]}{_cap[1]}' not in _arch:
        raise RuntimeError(
            f'GPU {_sm} is not supported by this torch build ({_arch}). On Kaggle pick '
            f'"GPU T4 x2" (sm_75) or "GPU P100" with a torch build that includes sm_60; '
            f'do not let pip upgrade torch (this notebook already installs with --no-deps).')


In [ ]:
CFG = dict(
    run_name=os.environ.get('B3_RUN', 'bm3d_3branch_200'),
    group=os.environ.get('B3_GROUP', 'bm3d_3branch_200'),
    bm3d_repo='tqhuyen/harvard-gf-denoise-benchmark-v2',
    bm3d_prefix='classical/bm3d/3375a321513938835d2c',
    labels_repo='tqhuyen/harvard-oct-glaucoma-200',
    data_dir=os.environ.get('B3_DATA_DIR', '/kaggle/temp/bm3d_200'),
    out_dir=os.environ.get('B3_OUT_DIR', '/kaggle/working/bm3d_3branch_200'),
    res3d=int(os.environ.get('B3_RES3D', '200')),
    res2d=224,
    store_res=200,
    enc2d=os.environ.get('B3_ENC2D', 'maxvit_tiny_rw_224'),
    enc3d_features=tuple(int(x) for x in os.environ.get('B3_ENC3D', '32,64,128,192').split(',')),
    latent=256,
    fusion=os.environ.get('B3_FUSION', 'crossgate'),
    view_indices=(0, 1),
    epochs=int(os.environ.get('B3_EPOCHS', '20')),
    patience=int(os.environ.get('B3_PATIENCE', '8')),
    lr=1e-4,
    wd=1e-4,
    effective_batch=16,
    num_workers=int(os.environ.get('B3_WORKERS', '0' if os.name == 'nt' else '2')),
    grad_checkpoint=os.environ.get('B3_GRAD_CKPT', '1') == '1',
    amp_dtype='float16',
    seed=42,
)
if SMOKE:
    CFG.update(data_dir=tempfile.mkdtemp(prefix='b3_data_'), out_dir=tempfile.mkdtemp(prefix='b3_out_'),
               res3d=8, res2d=8, store_res=8, enc2d='resnet18', enc3d_features=(8, 16, 32, 64), epochs=1,
               patience=2, effective_batch=2, num_workers=0, grad_checkpoint=False)
SPLITS = ('Training', 'Validation', 'Test')
Path(CFG['data_dir']).mkdir(parents=True, exist_ok=True)
Path(CFG['out_dir']).mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print('[config] res3d', CFG['res3d'], '| res2d', CFG['res2d'], '| enc2d', CFG['enc2d'],
      '| enc3d', CFG['enc3d_features'], '| fusion', CFG['fusion'],
      '| grad_ckpt', CFG['grad_checkpoint'])


In [ ]:
def _synthetic():
    rng = np.random.default_rng(0)
    for split in SPLITS:
        n = 8 if split == 'Training' else 4
        vols = rng.integers(0, 60, size=(n, 1, 8, 8, 8), dtype=np.uint8)
        vols[:, :, :, 2:6, :] = np.clip(vols[:, :, :, 2:6, :].astype(np.int16) + 90, 0, 255).astype(np.uint8)
        np.save(Path(CFG['data_dir']) / f'{split}_volumes.npy', vols)
        labs = np.array([0, 1] * (n // 2), dtype=np.int64)
        np.save(Path(CFG['data_dir']) / f'{split}_labels.npy', labs)

def _hf_download(repo, remote, local_dir):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=repo, filename=remote, repo_type='dataset',
                           token=os.environ.get('HF_TOKEN'), local_dir=str(local_dir))

def prepare_data():
    ready = all((Path(CFG['data_dir']) / f'{s}_volumes.npy').exists()
                and (Path(CFG['data_dir']) / f'{s}_labels.npy').exists()
                and (Path(CFG['data_dir']) / f'{s}_complete.json').exists() for s in SPLITS)
    if ready:
        print('[data] consolidated arrays already complete -> reuse', CFG['data_dir'])
        return
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get('HF_TOKEN'))
    names = [x.rfilename for x in api.dataset_info(CFG['bm3d_repo'], files_metadata=True).siblings]
    shard_dir = Path(CFG['data_dir']) / '_shards'
    shard_dir.mkdir(parents=True, exist_ok=True)
    for split in SPLITS:
        labels_path = Path(CFG['data_dir']) / f'{split}_labels.npy'
        if not labels_path.exists():
            src = _hf_download(CFG['labels_repo'], f'{split}_labels.npy', Path(CFG['data_dir']) / '_labels')
            np.save(labels_path, np.load(src))
        labels = np.load(labels_path)
        shards = sorted(n for n in names if n.startswith(f"{CFG['bm3d_prefix']}/volumes/{split}/") and n.endswith('.npy'))
        if not shards:
            raise RuntimeError(f'no BM3D shards found for {split}')
        total_bytes = sum(x.size for x in api.dataset_info(CFG['bm3d_repo'], files_metadata=True).siblings
                          if x.rfilename in shards)
        free = __import__('shutil').disk_usage(CFG['data_dir']).free
        if free < total_bytes + 2 * 1024**3:
            raise OSError(f'Need ~{(total_bytes + 2 * 1024**3) / 1024**3:.0f} GiB free, have {free / 1024**3:.1f} GiB')
        first = np.load(_hf_download(CFG['bm3d_repo'], shards[0], shard_dir), mmap_mode='r')
        sample = (1,) + tuple(first.shape[1:]) if first.ndim == 5 else tuple(first.shape[1:])
        out = np.lib.format.open_memmap(Path(CFG['data_dir']) / f'{split}_volumes.npy', mode='w+',
                                        dtype=np.uint8, shape=(len(labels),) + sample)
        cursor = 0
        for shard in [shards[0]] + shards[1:]:
            path = _hf_download(CFG['bm3d_repo'], shard, shard_dir)
            arr = np.load(path, mmap_mode='r')
            out[cursor:cursor + len(arr)] = arr
            out.flush()
            cursor += len(arr)
            Path(path).unlink(missing_ok=True)
            print(f'[data] {split} {cursor}/{len(labels)}', flush=True)
        if cursor != len(labels):
            raise RuntimeError(f'{split}: assembled {cursor} != labels {len(labels)}')
        with open(Path(CFG['data_dir']) / f'{split}_complete.json', 'w') as fh:
            json.dump({'complete': True, 'count': cursor, 'shape': list(out.shape)}, fh)
        del out
        print(f'[data] {split} done', flush=True)
    print('[data] BM3D 200^3 consolidated under', CFG['data_dir'])

def verify_data():
    for split in SPLITS:
        v = np.load(Path(CFG['data_dir']) / f'{split}_volumes.npy', mmap_mode='r')
        y = np.load(Path(CFG['data_dir']) / f'{split}_labels.npy')
        if v.dtype != np.uint8 or v.shape[-3:] != (CFG['store_res'],) * 3 or v.shape[0] != len(y):
            raise ValueError(f'{split}: bad shapes {v.shape} {y.shape}')
        print(f'[verify] {split}: {v.shape} {v.dtype} pos_rate={float(y.mean()):.3f}')

if SMOKE:
    _synthetic()
else:
    prepare_data()
verify_data()


In [ ]:
VIEW_FILES = {}
for split in SPLITS:
    src = Path(CFG['data_dir']) / f'{split}_volumes.npy'
    vp, dp = ft.build_views(src, res2d=CFG['res2d'])
    VIEW_FILES[split] = (vp, dp)
    print('[views]', split, vp.name, '|', dp.name)


In [ ]:
def make_dataset(split, train):
    return ft.FinalDataset(
        Path(CFG['data_dir']) / f'{split}_volumes.npy',
        Path(CFG['data_dir']) / f'{split}_labels.npy',
        res3d=CFG['res3d'], res2d=CFG['res2d'], seed=CFG['seed'], train=train)

TRAIN_DS = make_dataset('Training', True)
VAL_DS = make_dataset('Validation', False)
TEST_DS = make_dataset('Test', False)
print('[dataset] sizes', len(TRAIN_DS), len(VAL_DS), len(TEST_DS))


In [ ]:
def enable_checkpointing(model):
    from torch.utils.checkpoint import checkpoint
    for st in model.enc3d.stages:
        original = st.forward
        st.forward = (lambda orig: (lambda x: checkpoint(orig, x, use_reentrant=False)))(original)
    print('[model] gradient checkpointing enabled on 3D stages')

def build_model():
    model = cm.ControlsModel(
        n_2d=len(CFG['view_indices']), D=CFG['latent'], num_classes=2,
        enc2d=CFG['enc2d'], enc2d_pretrained=(not SMOKE),
        enc3d_features=CFG['enc3d_features'], view_indices=CFG['view_indices'],
        fusion=CFG['fusion'], use_3d=True, gate_fixed=False)
    if CFG['grad_checkpoint']:
        enable_checkpointing(model)
    return model


In [ ]:
def probe_batch(dataset):
    candidates = [8, 4, 2, 1] if not SMOKE else [2]
    chosen = 1
    for bs in candidates:
        model = build_model().to(DEVICE)
        loader = DataLoader(dataset, batch_size=bs, shuffle=False, num_workers=0)
        try:
            x, v, y = next(iter(loader))
            x, v, y = x.to(DEVICE), v.to(DEVICE), y.to(DEVICE)
            if USE_AMP:
                with torch.autocast('cuda', dtype=getattr(torch, CFG['amp_dtype'])):
                    loss = F.cross_entropy(model(x, v), y)
                loss.backward()
            else:
                loss = F.cross_entropy(model(x, v), y)
                loss.backward()
            chosen = bs
            del model, x, v, y, loss
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
            print('[probe] batch', bs, 'fits')
            break
        except RuntimeError as e:
            del model
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
            if 'out of memory' in str(e).lower():
                print('[probe] batch', bs, 'OOM -> retry smaller')
                continue
            raise
    return chosen

BS = probe_batch(TRAIN_DS)
GRAD_ACCUM = max(1, CFG['effective_batch'] // BS)
print('[probe] batch', BS, '| grad_accum', GRAD_ACCUM, '| effective', BS * GRAD_ACCUM)


In [ ]:
def init_wandb():
    import wandb
    return wandb.init(project='glaucoma-thesis', name=CFG['run_name'], group=CFG['group'],
                      config=CFG, resume='allow', id=CFG['run_name'])

def evaluate(model, dataset, batch_size):
    probs, labels, _ = ft.predict(model, dataset, batch_size, num_workers=CFG['num_workers'],
                                  amp_dtype=CFG['amp_dtype'])
    return fm.full_metrics(probs, labels, threshold=0.5), probs, labels

def train():
    model = build_model().to(DEVICE)
    n_par = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'[model] parameters {n_par:.2f}M')
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['epochs'])
    scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and CFG['amp_dtype'] == 'float16'))
    loader = DataLoader(TRAIN_DS, batch_size=BS, shuffle=True, num_workers=CFG['num_workers'],
                        pin_memory=(DEVICE.type == 'cuda'), drop_last=False,
                        generator=torch.Generator().manual_seed(CFG['seed']))
    run = init_wandb()
    history, best_val, best_state, bad = [], -1.0, None, 0
    for epoch in range(CFG['epochs']):
        TRAIN_DS.set_epoch(epoch)
        model.train()
        opt.zero_grad(set_to_none=True)
        running, steps, seen = 0.0, 0, 0
        t0 = time.time()
        for i, (x, v, y) in enumerate(loader):
            x, v, y = x.to(DEVICE, non_blocking=True), v.to(DEVICE, non_blocking=True), y.to(DEVICE)
            if USE_AMP:
                with torch.autocast('cuda', dtype=getattr(torch, CFG['amp_dtype'])):
                    loss = F.cross_entropy(model(x, v), y) / GRAD_ACCUM
                scaler.scale(loss).backward()
            else:
                loss = F.cross_entropy(model(x, v), y) / GRAD_ACCUM
                loss.backward()
            running += float(loss.detach()) * GRAD_ACCUM
            seen += y.numel()
            if (i + 1) % GRAD_ACCUM == 0:
                if USE_AMP:
                    scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if USE_AMP:
                    scaler.step(opt)
                    scaler.update()
                else:
                    opt.step()
                opt.zero_grad(set_to_none=True)
                steps += 1
        sched.step()
        val_m, _, _ = evaluate(model, VAL_DS, BS)
        train_loss = running / max(1, (i + 1))
        row = {'epoch': epoch + 1, 'train_loss': train_loss, 'sec': time.time() - t0, **val_m}
        history.append(row)
        run.log({'epoch': epoch + 1, 'train/loss': train_loss, 'train/samples': seen,
                 **{f'val/{k}': val_m[k] for k in ('acc', 'balanced_acc', 'f1', 'auc_roc', 'mcc')}}, step=epoch + 1)
        print(f"[train] ep{epoch+1:02d} loss={train_loss:.4f} "
              + ' '.join(f'{k}={val_m[k]:.4f}' for k in ('acc', 'balanced_acc', 'f1', 'auc_roc'))
              + f" ({row['sec']:.0f}s)")
        if val_m['auc_roc'] > best_val:
            best_val, bad = val_m['auc_roc'], 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG['patience']:
                print('[train] early stop'); break
    model.load_state_dict(best_state)
    report, test_probs, test_labels = ct.calibrated_report(
        model, VAL_DS, TEST_DS, BS, smoke=SMOKE, train=TRAIN_DS,
        num_workers=CFG['num_workers'], amp_dtype=CFG['amp_dtype'])
    run.log({'test/acc': report['test']['acc'], 'test/balanced_acc': report['test']['balanced_acc'],
             'test/f1': report['test']['f1'], 'test/auc_roc': report['test']['auc_roc'],
             'test/auc_pr': report['test']['auc_pr'], 'test/mcc': report['test']['mcc'],
             'test/threshold': report['threshold'], 'test/temperature': report['temperature']})
    out = Path(CFG['out_dir'])
    torch.save(best_state, out / 'best_weights.pt')
    with open(out / 'history.json', 'w') as fh:
        json.dump(history, fh, indent=2)
    with open(out / 'report.json', 'w') as fh:
        json.dump(report, fh, indent=2)
    run.summary.update({'val/best_auc': best_val, **{f'test/{k}': v for k, v in report['test'].items()}})
    run.finish()
    print('[report] test acc=%.4f f1=%.4f auc=%.4f' % (
        report['test']['acc'], report['test']['f1'], report['test']['auc_roc']))
    print('[done] artifacts in', out)
    return report

REPORT = train()
if SMOKE:
    print('B3_SMOKE_OK')


## After the run
- Artifacts are under `/kaggle/working/...` - use **Save Version** so Kaggle persists the
  output (or `kaggle datasets create` from the output folder).
- `best_weights.pt`, `history.json`, `report.json` are logged/uploaded to W&B
  (project `glaucoma-thesis`) as required by the repo protocol.
- To compare against raw/bilateral, rerun with a different `B3_DATA_DIR` and `B3_GROUP`.
